# v3 JET vs NO-JET on Colab (free T4)
1. Upload the `archive/` folder (Kaggle download) to Drive: `MyDrive/jetdata/archive/`
   (needs `archive/audio/`, `archive/sample_meta.csv`; `env_audio/` optional).
2. Runtime -> Change runtime type -> T4 GPU. 3. Run all. Model saves back to Drive.
Target to beat (their CNN, same split, AP): test 98.60, env ~89.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
import glob
CANDS = glob.glob('/content/drive/MyDrive/jetdata/**/sample_meta.csv', recursive=True)
print('found:', CANDS or 'NONE - upload archive/ to Drive/MyDrive/jetdata/ first')
assert CANDS, 'Upload the archive folder to Drive/MyDrive/jetdata/ first'
META = CANDS[0]; ROOT = Path(META).parent
AUD = ROOT/'archive'/'audio' if (ROOT/'archive'/'audio').exists() else ROOT/'audio'
OUT = Path('/content/drive/MyDrive/jetdata/output'); OUT.mkdir(exist_ok=True)
print('meta:', META, '| audio:', AUD, '| wavs:', len(list(AUD.rglob('*.wav'))), '| out:', OUT)

In [ ]:
import numpy as np, torch, torch.nn as nn, torchaudio
import torch.nn.functional as TF
SR, N_MELS = 16000, 64
MEL = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=1024, hop_length=256, n_mels=N_MELS)
DB = torchaudio.transforms.AmplitudeToDB()
def logmel(w):
    return DB(MEL(w.unsqueeze(0) if w.dim()==1 else w)).squeeze(0)
class JetCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d((4,8)))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(64*4*8,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,1))
    def forward(self,x): return self.fc(self.net(x)).squeeze(1)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
EPOCHS, BATCH, LR = 20, 64, 3e-4
random.seed(0); torch.manual_seed(0)
split = {r['filename']: r['train-test'] for r in csv.DictReader(open(META))}
sub = {r['filename']: r['subclass'] for r in csv.DictReader(open(META))}
tr, te = {1:[],0:[]}, {1:[],0:[]}
for p in AUD.rglob('*.wav'):
    s = sub.get(p.name)
    if s not in ('3','0'): continue
    (te if split.get(p.name)=='test' else tr)[1 if s=='3' else 0].append(p)
print('train JET/NOJET:', len(tr[1]), len(tr[0]), '| test:', len(te[1]), len(te[0]))
def wins(p, per, aug=False):
    w,sr = torchaudio.load(str(p))
    if w.shape[0]>1: w = w.mean(0, keepdim=True)
    if sr!=SR: w = torchaudio.functional.resample(w,sr,SR)
    w = w.squeeze(0); n = 5*SR
    if len(w)<n: return [logmel(TF.pad(w,(0,n-len(w)))).numpy()]
    idx = [0] if len(w)<=12*SR else list(range(0,len(w)-n+1,n//2))[:per]
    out=[]
    for i in idx[:per]:
        c = w[i:i+n]
        if aug:
            if random.random()<0.5: c = c*random.uniform(0.7,1.3)
            if random.random()<0.5: c = c+torch.randn_like(c)*random.uniform(0.0,0.05)
        out.append(logmel(c).numpy())
    return out
def build(fm, pj, pn, cj, cn, aug=False):
    X,y=[],[]
    for lab,(per,cap) in ((1,(pj,cj)),(0,(pn,cn))):
        F=fm[lab][:]; random.shuffle(F)
        for p in F:
            if sum(v==lab for v in y)>=cap: break
            for f in wins(p,per,aug): X.append(f); y.append(lab)
    return np.stack(X)[:,None,:,:], np.array(y,dtype=np.float32)
Xtr,ytr = build(tr,4,2,2500,2500)
Xa,ya = build({1:tr[1],0:[]},4,2,1500,0,aug=True)
Xtr=np.concatenate([Xtr,Xa]); ytr=np.concatenate([ytr,ya])
Xte,yte = build(te,4,2,10**9,10**9)
print('train',Xtr.shape,'test',Xte.shape)

In [ ]:
from sklearn.metrics import average_precision_score
dev='cuda' if torch.cuda.is_available() else 'cpu'
model=JetCNN().to(dev); pr=(ytr==1).sum()/len(ytr)
opt=torch.optim.Adam(model.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS)
fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1-pr)/pr]).to(dev))
Xt=torch.tensor(Xtr,dtype=torch.float32); yt=torch.tensor(ytr,dtype=torch.float32); Xe=torch.tensor(Xte,dtype=torch.float32)
best=-1; hist=[]
for ep in range(EPOCHS):
    model.train(); perm=torch.randperm(len(Xt)); tot=0
    for i in range(0,len(perm),BATCH):
        b=perm[i:i+BATCH]; opt.zero_grad()
        loss=fn(model(Xt[b].to(dev)),yt[b].to(dev)); loss.backward(); opt.step(); tot+=loss.item()*len(b)
    sch.step(); model.eval()
    with torch.no_grad():
        pa=((torch.sigmoid(model(Xt.to(dev)))>0.5).cpu().numpy()==ytr).mean()
        pte=torch.sigmoid(model(Xe.to(dev))).cpu().numpy()
        pe=((pte>0.5)==yte).mean(); ap=average_precision_score(yte,pte)
    hist.append((tot/len(Xt),pa,pe,ap))
    if ap>best: best=ap; torch.save(model.state_dict(),'/content/drive/MyDrive/jetdata/output/v3_jet.pt'); flag=' <-- BEST'
    else: flag=''
    print(f'ep{ep+1}/{EPOCHS} loss={tot/len(Xt):.4f} train={pa:.3f} test={pe:.3f} AP={ap:.4f}{flag}')
print(f'BEST AP={best:.4f} (theirs CNN test 0.9860)')

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
ep=list(range(1,len(hist)+1))
plt.figure(figsize=(10,5))
plt.plot(ep,[h[1] for h in hist],'o-',label='train acc')
plt.plot(ep,[h[2] for h in hist],'s-',label='test acc')
plt.plot(ep,[h[3] for h in hist],'^-',label='test AP')
plt.xlabel('epoch'); plt.ylabel('score'); plt.ylim(0,1.03)
plt.title('v3 Kaggle-GPU (aerosonic, best-ckpt AP)'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/content/drive/MyDrive/jetdata/output/acc_v3_kaggle.png',dpi=150)
print('saved /content/drive/MyDrive/jetdata/output/v3_jet.pt + acc_v3_kaggle.png')